In [6]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import warnings
warnings.filterwarnings('ignore')

In [7]:
# Carregar dataset espelhado
df = pd.read_csv('../data/annotated/annotations_mirrored_batch.csv')

# Separar EN e PT
en = df[df['language'] == 'EN'].copy()
pt = df[df['language'] == 'PT'].copy()

print(f"Pares EN: {len(en)}")
print(f"Pares PT: {len(pt)}")
print(f"\nColunas: {df.columns.tolist()}")

Pares EN: 10
Pares PT: 10

Colunas: ['id', 'language', 'category', 'prompt', 'response', 'faithfulness', 'relevance', 'fluency', 'completeness', 'safety', 'hallucination_type', 'feedback', 'annotator']


In [8]:
#Verificar alinhamento dos pares

# Extrair o ID base (sem -EN ou -PT)
en['base_id'] = en['id'].str.replace('-EN', '')
pt['base_id'] = pt['id'].str.replace('-PT', '')

# Ordenar por base_id para garantir alinhamento
en = en.sort_values('base_id').reset_index(drop=True)
pt = pt.sort_values('base_id').reset_index(drop=True)

# Verificar alinhamento
print("Pares alinhados:")
for i in range(len(en)):
    print(f"  {en.loc[i,'base_id']}: EN={en.loc[i,'id']} | PT={pt.loc[i,'id']}")

Pares alinhados:
  MIR-A1: EN=MIR-A1-EN | PT=MIR-A1-PT
  MIR-A2: EN=MIR-A2-EN | PT=MIR-A2-PT
  MIR-AM1: EN=MIR-AM1-EN | PT=MIR-AM1-PT
  MIR-C1: EN=MIR-C1-EN | PT=MIR-C1-PT
  MIR-F1: EN=MIR-F1-EN | PT=MIR-F1-PT
  MIR-F2: EN=MIR-F2-EN | PT=MIR-F2-PT
  MIR-F3: EN=MIR-F3-EN | PT=MIR-F3-PT
  MIR-F4: EN=MIR-F4-EN | PT=MIR-F4-PT
  MIR-R1: EN=MIR-R1-EN | PT=MIR-R1-PT
  MIR-R2: EN=MIR-R2-EN | PT=MIR-R2-PT


In [9]:
#Calcular kappa por dimensão
dimensoes = ['faithfulness', 'relevance', 'fluency', 'completeness', 'safety']
resultados = {}

print("=" * 60)
print(f"{'Dimensão':<15} {'Kappa Simples':>14} {'Kappa Ponderado':>16} {'Status':>8}")
print("=" * 60)

for dim in dimensoes:
    scores_en = en[dim].tolist()
    scores_pt = pt[dim].tolist()

    if len(set(scores_en + scores_pt)) == 1:
        print(f"{dim:<15} {'— concordância perfeita —':>32}  ✅")
        resultados[dim] = {'kappa': 1.0, 'kappa_w': 1.0, 'status': 'perfeito'}
    else:
        try:
            kappa   = cohen_kappa_score(scores_en, scores_pt)
            kappa_w = cohen_kappa_score(scores_en, scores_pt, weights='quadratic')
            status  = "✅" if kappa_w >= 0.60 else "⚠️ revisar"
            print(f"{dim:<15} {kappa:>14.3f} {kappa_w:>16.3f} {status:>8}")
            resultados[dim] = {'kappa': round(kappa,3), 'kappa_w': round(kappa_w,3), 'status': 'ok'}
        except Exception as e:
            print(f"{dim:<15} {'Erro: ' + str(e):>32}")
            resultados[dim] = {'kappa': None, 'kappa_w': None, 'status': 'erro'}

print("=" * 60)

Dimensão         Kappa Simples  Kappa Ponderado   Status
faithfulness           — concordância perfeita —  ✅
relevance                0.000            0.000 ⚠️ revisar
fluency                  0.000            0.000 ⚠️ revisar
completeness            -0.111           -0.111 ⚠️ revisar
safety                 — concordância perfeita —  ✅


In [10]:
#Kappa para hallucination_type

print("\nKappa para hallucination_type (categórico):")
print("-" * 45)

hal_en = en['hallucination_type'].tolist()
hal_pt = pt['hallucination_type'].tolist()

if len(set(hal_en + hal_pt)) == 1:
    print("Todos os pares com mesmo tipo — concordância perfeita.")
else:
    kappa_hal = cohen_kappa_score(hal_en, hal_pt)
    status = "✅" if kappa_hal >= 0.60 else "⚠️ revisar"
    print(f"Cohen's Kappa simples: {kappa_hal:.3f}  {status}")
    resultados['hallucination_type'] = {
        'kappa': round(kappa_hal, 3),
        'kappa_w': None,
        'status': 'ok'
    }

print()
for i in range(len(en)):
    en_hal = en.loc[i, 'hallucination_type']
    pt_hal = pt.loc[i, 'hallucination_type']
    if en_hal != pt_hal:
        print(f"  Discordância: {en.loc[i,'base_id']} → EN={en_hal} | PT={pt_hal}")


Kappa para hallucination_type (categórico):
---------------------------------------------
Cohen's Kappa simples: -0.111  ⚠️ revisar

  Discordância: MIR-A2 → EN=none | PT=extrinsic
  Discordância: MIR-F4 → EN=extrinsic | PT=none


In [11]:
#Analisando as discordâncias

print("\nDiscordâncias por dimensão:")
print("=" * 55)

todas_discordancias = []

for dim in dimensoes:
    disc = []
    for i in range(len(en)):
        if en.loc[i, dim] != pt.loc[i, dim]:
            disc.append({
                'par': en.loc[i, 'base_id'],
                'dimensao': dim,
                'score_en': en.loc[i, dim],
                'score_pt': pt.loc[i, dim],
                'diferenca': abs(en.loc[i, dim] - pt.loc[i, dim])
            })
            todas_discordancias.append(disc[-1])

    if disc:
        print(f"\n{dim.upper()} — {len(disc)} discordância(s):")
        for d in disc:
            direcao = "PT < EN" if d['score_pt'] < d['score_en'] else "PT > EN"
            print(f"  {d['par']}: EN={d['score_en']} | PT={d['score_pt']} | {direcao}")
    else:
        print(f"\n{dim.upper()} — nenhuma discordância ✅")

print(f"\nTotal de discordâncias: {len(todas_discordancias)} em {len(en)*len(dimensoes)} comparações")
print(f"Taxa de concordância: {(1 - len(todas_discordancias)/(len(en)*len(dimensoes)))*100:.1f}%")


Discordâncias por dimensão:

FAITHFULNESS — nenhuma discordância ✅

RELEVANCE — 1 discordância(s):
  MIR-C1: EN=5 | PT=4 | PT < EN

FLUENCY — 1 discordância(s):
  MIR-A2: EN=5 | PT=4 | PT < EN

COMPLETENESS — 2 discordância(s):
  MIR-F2: EN=5 | PT=4 | PT < EN
  MIR-F4: EN=4 | PT=5 | PT > EN

SAFETY — nenhuma discordância ✅

Total de discordâncias: 4 em 50 comparações
Taxa de concordância: 92.0%


In [12]:
#Interpretando as discordâncias

print("\nINTERPRETAÇÃO DAS DISCORDÂNCIAS")
print("=" * 55)

interpretacoes = {
    'MIR-F2': "PT omite escala Fahrenheit — menos relevante para público lusófono.",
    'MIR-F4': "PT mais completo — detalhe histórico da construção de Brasília incluído.",
    'MIR-A2': "PT não menciona AVC explicitamente — omissão menor mas documentável.",
    'MIR-R2': "PT: '0,5 ovo por porção' soa artificial — fluency naturalmente menor.",
    'MIR-C1': "PT: Meetup tem baixa penetração em PT/BR — relevance reduzida.",
}

for par, motivo in interpretacoes.items():
    print(f"\n  {par}: {motivo}")

print("""
CONCLUSÃO:
Todas as discordâncias têm justificativa linguística ou cultural
clara — não são erros de anotação, são diferenças sistemáticas
entre EN e PT que um avaliador bilíngue nativo consegue capturar.
Isso valida o uso de anotação bilíngue separada em vez de
tradução direta dos labels.
""")


INTERPRETAÇÃO DAS DISCORDÂNCIAS

  MIR-F2: PT omite escala Fahrenheit — menos relevante para público lusófono.

  MIR-F4: PT mais completo — detalhe histórico da construção de Brasília incluído.

  MIR-A2: PT não menciona AVC explicitamente — omissão menor mas documentável.

  MIR-R2: PT: '0,5 ovo por porção' soa artificial — fluency naturalmente menor.

  MIR-C1: PT: Meetup tem baixa penetração em PT/BR — relevance reduzida.

CONCLUSÃO:
Todas as discordâncias têm justificativa linguística ou cultural
clara — não são erros de anotação, são diferenças sistemáticas
entre EN e PT que um avaliador bilíngue nativo consegue capturar.
Isso valida o uso de anotação bilíngue separada em vez de
tradução direta dos labels.



In [13]:
#Salvar relatório

relatorio = []
for dim, res in resultados.items():
    relatorio.append({
        'dimensao': dim,
        'kappa_simples': res['kappa'],
        'kappa_ponderado': res.get('kappa_w'),
        'status': res['status']
    })

df_rel = pd.DataFrame(relatorio)
df_rel.to_csv('../data/inter_annotator/kappa_report.csv', index=False)
print(df_rel.to_string(index=False))
print("\n✅ Relatório salvo em data/inter_annotator/kappa_report.csv")

          dimensao  kappa_simples  kappa_ponderado   status
      faithfulness          1.000            1.000 perfeito
         relevance          0.000            0.000       ok
           fluency          0.000            0.000       ok
      completeness         -0.111           -0.111       ok
            safety          1.000            1.000 perfeito
hallucination_type         -0.111              NaN       ok

✅ Relatório salvo em data/inter_annotator/kappa_report.csv
